In [19]:
import altair as alt
import pandas as pd
import numpy as np
import streamlit as st

In [4]:
# Load clean dataset
df = pd.read_csv("../data/clean_data/simpsons_episodes_clean.csv")
df["original_air_date"] = pd.to_datetime(df["original_air_date"])

# Derived columns
df["era"] = pd.cut(
    df["season"],
    bins=[0, 8, 18, 27],
    labels=["Golden Age (S1-8)", "Middle (S9-18)", "Later (S19-27)"],
)

## Question 1

In [5]:
# Box plot per season + mean line (colored by era)
era_domain = ["Golden Age (S1-8)", "Middle (S9-18)", "Later (S19-27)"]
era_colors = ["#4c78a8", "#f58518", "#e45756"]

q1c_era_box = (
    alt.Chart(df)
    .mark_boxplot(size=15)
    .encode(
        x=alt.X("season:O", title="Season", axis=alt.Axis(labelAngle=0)),
        y=alt.Y(
            "imdb_rating:Q",
            title="IMDb Rating",
            scale=alt.Scale(domain=[4, 10]),
        ),
        color=alt.Color(
            "era:N",
            scale=alt.Scale(domain=era_domain, range=era_colors),
            legend=alt.Legend(title="Era"),
        ),
    )
)

q1c_era_mean = (
    alt.Chart(df)
    .mark_line(color="black", strokeWidth=2)
    .encode(
        x=alt.X("season:O"),
        y=alt.Y("mean(imdb_rating):Q"),
    )
)

q1c_era_points = q1c_era_mean.mark_point(color="black", filled=True, size=40)

(q1c_era_box + q1c_era_mean + q1c_era_points).properties(
    width=700, height=350, title="IMDb Rating per Season (Era Colors)"
)

alt.LayerChart(...)

## Question 2

In [6]:
# Box plot per season + mean line (colored by era)
era_domain = ["Golden Age (S1-8)", "Middle (S9-18)", "Later (S19-27)"]
era_colors = ["#4c78a8", "#f58518", "#e45756"]

q2c_era_box = (
    alt.Chart(df)
    .mark_boxplot(size=15)
    .encode(
        x=alt.X("season:O", title="Season", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (millions)"),
        color=alt.Color(
            "era:N",
            scale=alt.Scale(domain=era_domain, range=era_colors),
            legend=alt.Legend(title="Era"),
        ),
    )
)

q2c_era_mean = (
    alt.Chart(df)
    .mark_line(color="black", strokeWidth=2)
    .encode(
        x=alt.X("season:O"),
        y=alt.Y("mean(us_viewers_in_millions):Q"),
    )
)

q2c_era_points = q2c_era_mean.mark_point(color="black", filled=True, size=40)

(q2c_era_box + q2c_era_mean + q2c_era_points).properties(
    width=700, height=350, title="US Viewers per Season (Era Colors)"
)

alt.LayerChart(...)

## Question 3

In [8]:
# Scatter (flipped axes: viewers on x, rating on y)
era_domain = ["Golden Age (S1-8)", "Middle (S9-18)", "Later (S19-27)"]
era_colors = ["#4c78a8", "#f58518", "#e45756"]
era_scale = alt.Scale(domain=era_domain, range=era_colors)

q3f_scatter = (
    alt.Chart(df)
    .mark_circle(size=50, opacity=0.5)
    .encode(
        x=alt.X("us_viewers_in_millions:Q", title="US Viewers (millions)"),
        y=alt.Y("imdb_rating:Q", title="IMDb Rating", scale=alt.Scale(zero=False)),
        color=alt.Color("era:N", scale=era_scale, title="Era"),
        tooltip=[
            "title:N",
            "season:O",
            "era:N",
            "imdb_rating:Q",
            "us_viewers_in_millions:Q",
        ],
    )
)

(q3f_scatter).properties(
    width=500, height=400, title="Viewers vs Rating (Regression per Era, flipped axes)"
)

alt.Chart(...)

## Question 4

In [10]:
df_thu_sun = df[df["weekday"].isin(["Thursday", "Sunday"])].copy()

In [11]:
# Box plot Thursday vs Sunday
# Thursday = seasons 2-5 (Golden Age), Sunday = seasons 1+6-27 (spans all eras)
thu_sun_labels = ["Thursday\n(Golden Age)", "Sunday\n(Middle + Later)"]

df_thu_sun["weekday_label"] = df_thu_sun["weekday"].map(
    {
        "Thursday": thu_sun_labels[0],
        "Sunday": thu_sun_labels[1],
    }
)

alt.Chart(df_thu_sun).mark_boxplot(size=60).encode(
    x=alt.X(
        "weekday_label:N",
        sort=thu_sun_labels,
        title="Day of the Week",
        axis=alt.Axis(labelAngle=0),
    ),
    y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (millions)"),
    color=alt.Color(
        "weekday_label:N",
        sort=thu_sun_labels,
        legend=None,
        scale=alt.Scale(domain=thu_sun_labels, range=["#4c78a8", "#e45756"]),
    ),
).properties(width=400, height=400, title="US Viewers: Thursday vs Sunday")

alt.Chart(...)

## Question 5

In [16]:
season_avg = df.groupby("season")["us_viewers_in_millions"].transform("mean")
df["viewers_normalized"] = df["us_viewers_in_millions"] / season_avg

# Mean normalized viewership by episode position, split by era
era_domain = ["Golden Age (S1-8)", "Middle (S9-18)", "Later (S19-27)"]
era_colors = ["#4c78a8", "#f58518", "#e45756"]

pos_era_norm = (
    df.groupby(["era", "number_in_season"])["viewers_normalized"]
    .agg(["mean", "count"])
    .reset_index()
)
pos_era_norm = pos_era_norm[pos_era_norm["count"] >= 3]

q5f_lines = (
    alt.Chart(pos_era_norm)
    .mark_line(strokeWidth=2)
    .encode(
        x=alt.X(
            "number_in_season:O",
            title="Episode Position in Season",
            axis=alt.Axis(labelAngle=0),
        ),
        y=alt.Y(
            "mean:Q",
            title="Viewers (relative to season average)",
            axis=alt.Axis(format=".0%"),
        ),
        color=alt.Color(
            "era:N",
            scale=alt.Scale(domain=era_domain, range=era_colors),
            title="Era",
        ),
        tooltip=[
            alt.Tooltip("era:N", title="Era"),
            alt.Tooltip("number_in_season:O", title="Episode"),
            alt.Tooltip("mean:Q", title="Relative Viewers", format=".1%"),
            alt.Tooltip("count:Q", title="Seasons"),
        ],
    )
)

q5f_points = (
    alt.Chart(pos_era_norm)
    .mark_point(filled=True, size=35)
    .encode(
        x=alt.X("number_in_season:O", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("mean:Q"),
        color=alt.Color(
            "era:N",
            scale=alt.Scale(domain=era_domain, range=era_colors),
            title="Era",
        ),
    )
)

q5f_ref = (
    alt.Chart(pd.DataFrame({"y": [1.0]}))
    .mark_rule(color="black", strokeDash=[4, 4], opacity=0.5)
    .encode(y="y:Q")
)

(q5f_lines + q5f_points + q5f_ref).properties(
    width=700,
    height=350,
    title="Within-Season Viewership by Episode Position (Normalized, Split by Era)",
)

C:\Users\20203666\AppData\Local\Temp\ipykernel_16700\2068121792.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(["era", "number_in_season"])["viewers_normalized"]


alt.LayerChart(...)

In [21]:
# Mean viewership by episode position, split by era (NOT normalized, absolute values)
pos_era_abs = (
    df.groupby(["era", "number_in_season"])["us_viewers_in_millions"]
    .agg(["mean", "count"])
    .reset_index()
)
pos_era_abs = pos_era_abs[pos_era_abs["count"] >= 3]

q5f_abs_lines = (
    alt.Chart(pos_era_abs)
    .mark_line(strokeWidth=2)
    .encode(
        x=alt.X(
            "number_in_season:O",
            title="Episode Position in Season",
            axis=alt.Axis(labelAngle=0),
        ),
        y=alt.Y("mean:Q", title="Mean US Viewers (millions)"),
        color=alt.Color(
            "era:N",
            scale=alt.Scale(domain=era_domain, range=era_colors),
            title="Era",
        ),
        tooltip=[
            alt.Tooltip("era:N", title="Era"),
            alt.Tooltip("number_in_season:O", title="Episode"),
            alt.Tooltip("mean:Q", title="Mean Viewers (M)", format=".2f"),
            alt.Tooltip("count:Q", title="Seasons"),
        ],
    )
)

q5f_abs_points = (
    alt.Chart(pos_era_abs)
    .mark_point(filled=True, size=35)
    .encode(
        x=alt.X("number_in_season:O", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("mean:Q"),
        color=alt.Color(
            "era:N",
            scale=alt.Scale(domain=era_domain, range=era_colors),
            title="Era",
        ),
    )
)

(q5f_abs_lines + q5f_abs_points).properties(
    width=700,
    height=350,
    title="Within-Season Viewership by Episode Position (Absolute, Split by Era)",
)

C:\Users\20203666\AppData\Local\Temp\ipykernel_16700\1566993480.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(["era", "number_in_season"])["us_viewers_in_millions"]


alt.LayerChart(...)

In [22]:
# Mean viewership by episode position (aggregated across all seasons)
pos_stats = (
    df.groupby("number_in_season")["us_viewers_in_millions"]
    .agg(["mean", "count"])
    .reset_index()
)
pos_stats = pos_stats[pos_stats["count"] >= 10]

overall_mean = df["us_viewers_in_millions"].mean()

q5_agg_line = (
    alt.Chart(pos_stats)
    .mark_line(point=True, strokeWidth=2)
    .encode(
        x=alt.X(
            "number_in_season:O",
            title="Episode Position in Season",
            axis=alt.Axis(labelAngle=0),
        ),
        y=alt.Y("mean:Q", title="Mean US Viewers (millions)"),
        color=alt.value("#4c78a8"),
        tooltip=[
            alt.Tooltip("number_in_season:O", title="Episode"),
            alt.Tooltip("mean:Q", title="Mean Viewers (M)", format=".2f"),
            alt.Tooltip("count:Q", title="Seasons with this position"),
        ],
    )
)

q5_agg_ref = (
    alt.Chart(pd.DataFrame({"y": [overall_mean]}))
    .mark_rule(color="black", strokeDash=[4, 4], opacity=0.5)
    .encode(y="y:Q")
)

(q5_agg_line + q5_agg_ref).properties(
    width=600,
    height=350,
    title="Mean Viewership by Episode Position (Aggregated Across Seasons)",
)

alt.LayerChart(...)

In [23]:
# Mean viewership by episode position, split by era (NOT normalized, absolute values)
pos_era_abs = (
    df.groupby(["era", "number_in_season"])["us_viewers_in_millions"]
    .agg(["mean", "count"])
    .reset_index()
)
pos_era_abs = pos_era_abs[pos_era_abs["count"] >= 3]

# Era averages for horizontal reference lines
era_avgs = df.groupby("era")["us_viewers_in_millions"].mean().reset_index()
era_avgs.columns = ["era", "era_mean"]

era_domain = ["Golden Age (S1-8)", "Middle (S9-18)", "Later (S19-27)"]
era_colors = ["#4c78a8", "#f58518", "#e45756"]

q5f_abs_lines = (
    alt.Chart(pos_era_abs)
    .mark_line(strokeWidth=2)
    .encode(
        x=alt.X(
            "number_in_season:O",
            title="Episode Position in Season",
            axis=alt.Axis(labelAngle=0),
        ),
        y=alt.Y("mean:Q", title="Mean US Viewers (millions)"),
        color=alt.Color(
            "era:N",
            scale=alt.Scale(domain=era_domain, range=era_colors),
            title="Era",
        ),
        tooltip=[
            alt.Tooltip("era:N", title="Era"),
            alt.Tooltip("number_in_season:O", title="Episode"),
            alt.Tooltip("mean:Q", title="Mean Viewers (M)", format=".2f"),
            alt.Tooltip("count:Q", title="Seasons"),
        ],
    )
)

q5f_abs_points = (
    alt.Chart(pos_era_abs)
    .mark_point(filled=True, size=35)
    .encode(
        x=alt.X("number_in_season:O", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("mean:Q"),
        color=alt.Color(
            "era:N",
            scale=alt.Scale(domain=era_domain, range=era_colors),
            title="Era",
        ),
    )
)

q5f_abs_era_refs = (
    alt.Chart(era_avgs)
    .mark_rule(strokeDash=[4, 4], opacity=0.6, strokeWidth=1.5)
    .encode(
        y=alt.Y("era_mean:Q"),
        color=alt.Color(
            "era:N",
            scale=alt.Scale(domain=era_domain, range=era_colors),
            legend=None,
        ),
    )
)

(q5f_abs_lines + q5f_abs_points + q5f_abs_era_refs).properties(
    width=700,
    height=350,
    title="Within-Season Viewership by Episode Position (Absolute, Split by Era)",
)


C:\Users\20203666\AppData\Local\Temp\ipykernel_16700\2542185071.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(["era", "number_in_season"])["us_viewers_in_millions"]
C:\Users\20203666\AppData\Local\Temp\ipykernel_16700\2542185071.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  era_avgs = df.groupby("era")["us_viewers_in_millions"].mean().reset_index()


alt.LayerChart(...)